# Toy Data Evaluation

In [ ]:
import Evaluation as E

recall = E.Recall([5,14,19,6,11,3,2,10,200,15,17,12,131,89],[90,11,5,6,10,60,14,89,19,200],toyData=True,N=30)
f1_score = E.F1_score([5,14,19,6,11,3,2,10,200,15,17,12,131,89],[90,11,5,6,10,60,14,89,19,200],toyData=True,N=30)
precission = E.Precision([5,14,19,6,11,3,2,10,200,15,17,12,131,89],[90,11,5,6,10,60,14,89,19,200],toyData=True,N=30)
ap = E.AP([5,14,19,6,11,3,2,10,200,15,17,12,131,89],[90,11,5,6,10,60,14,89,19,200],toyData=True,N=30)

In [ ]:
print("precission : ",precission.result_evaluation)
print("=="*99)
print("Recall : ",recall.result_evaluation)
print("=="*99)
print("f1_score : ",f1_score.result_evaluation)
print("=="*99)
print("AP : ",ap.result_evaluation)

# Helper


In [ ]:
import pandas as pd

def skenario_pertama_tversky_index(data) :
    from itertools import product

    skenario_recall = data[0.7][30][20]

    gamma = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]


    ti_gamma_alpha_recall = []
    # Mengakses parameter gamma
    ti_mean_gamma_overall_fold_recall = []
    for gamma_index_1, gamma_index_2 in product(gamma,gamma) :
        
        skenario = skenario_recall[gamma_index_1][gamma_index_2]
        
        ti_mean_gamma_fold_recall = []

        for fold in skenario :

            # Tahap 1
            ti_sum_matrix_recall = sum([i[99] for i in fold]) 
            ti_mean_gamma_fold_recall.append(ti_sum_matrix_recall/len(fold))
        
        # Tahap 3
        ti_mean_matrix_per_fold_recall = sum(ti_mean_gamma_fold_recall)/len(skenario)
        
        ti_mean_gamma_fold_recall.append(ti_mean_matrix_per_fold_recall)
        ti_mean_gamma_overall_fold_recall.append(ti_mean_gamma_fold_recall)
    
    return {
        "result" : ti_mean_gamma_fold_recall,
        "mean" : ti_mean_gamma_overall_fold_recall,
    }

def visualisasi_skenario_pertama_tversky_index(data) :
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np

    alpha = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    # Ambil kolom rata-rata dari ti_mean_gamma_overall_fold_recall
    rata_rata = np.array(data)[:, -1]

    # Ubah menjadi matriks 11x11 sesuai alpha_1 dan alpha_2
    heatmap_matrix = rata_rata.reshape(len(alpha), len(alpha))

    plt.figure(figsize=(10, 8))
    sns.heatmap(heatmap_matrix, annot=True, fmt=".2f", cmap="viridis", 
                xticklabels=alpha, yticklabels=alpha, cbar_kws={'label': 'Recall rata-rata'})
    plt.xlabel(r'$\alpha_2$')
    plt.ylabel(r'$\alpha_1$')
    plt.title(r'Heatmap Recall rata-rata: $\alpha_1$ vs $\alpha_2$')
    plt.show()

def get_highest_value(dataframe : pd.DataFrame) -> dict :
    candidate = {
    "key" : "",
    "value" : 0,
    }

    for i in dataframe.T :
        rata_rata = dataframe.T[i]["rata-rata"]

        if rata_rata >= candidate["value"] :
            candidate = {
                "key" : i,
                "value" : rata_rata,
            }

    print("File :",candidate["key"],",","Nilai :",candidate["value"])


## Kedua TI

In [ ]:
def skenario_kedua_tversky_index(skenario : object,*, alpha_1 = 0.1, alpha_2 = 0) :
    import numpy as np

    n = 100
    k_user = [5,10,15,18,20,25,30,40,50,100,200]


    ti_k_user_fold = []
    ti_mean_k_user_fold = []

    # Mengakses parameter k_user
    for k_user_index in k_user :

        skenario_kedua_k_user = skenario[0.7][k_user_index][20][alpha_1][alpha_2]
        ti_mean_k_user_per_fold = []

        for fold in skenario_kedua_k_user :
            # Tahap 1 : menjumlahkan evaluasi setiap fold
            ti_sum_k_user_per_fold = sum(j[99] for j in fold)
            ti_mean_k_user_per_fold += [ti_sum_k_user_per_fold/len(skenario_kedua_k_user)]
        
        # Tahap 3 : menghitung rata-rata setiap fold dengan menjumlahkan nilai evaluasi setiap pengguna dibagi dengan banyaknya pengguna
        mean_k_user_per_fold = sum(ti_mean_k_user_per_fold)/len(skenario_kedua_k_user)
        
        ti_mean_k_user_per_fold += [mean_k_user_per_fold]
        ti_k_user_fold += [ti_mean_k_user_per_fold]
        ti_mean_k_user_fold += [[5,k_user_index,mean_k_user_per_fold]]

    return {
        "mean" : ti_k_user_fold,
        "result" : ti_mean_k_user_fold,
    }

def df_skenario_kedua_tversky_index(dataProcess, type = "result"):
    import pandas as pd

    k_user = [5,10,15,18,20,25,30,40,50,100,200]

    return pd.DataFrame(dataProcess[type],index=[ f"k_user_{i}" for i in k_user],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])

def visualisation_skenario_kedua_tversky_index_per_fold(dataProcess, type, title) :
    import matplotlib.pyplot as plt
    k_user = [5,10,15,18,20,25,30,40,50,100,200]

    if type == "mean" :
        # Definisikan markers dan colors
        # markers = ['o', 's', '^', 'D', 'o']
        colors = ['b', 'g', 'r', 'c', 'm']

        plt.figure(figsize=(10, 6))

        for fold_idx in range(5):
            per_fold = [dataProcess[type][k][fold_idx] for k in range(len(k_user))]
            plt.plot(
                k_user,
                per_fold,
                marker="o",
                color=colors[fold_idx],
                label=f'Fold-{fold_idx+1}'
            )
        plt.legend()

    elif type == "result" :
        import pandas as pd
        df = pd.DataFrame(dataProcess[type], index=[i for i in range(1, len(dataProcess[type]) + 1)], columns=["k_user", "k_item", title])

        import matplotlib.pyplot as plt

        plt.figure(figsize=(10, 6))
        plt.plot(k_user, df[title], marker='o')
    
    plt.xlabel(r'$k^{user}$')
    plt.ylabel(title)
    plt.title(title+r' per Fold untuk Setiap $k^{user}$')
    plt.grid(True)
    plt.show()

## Ketiga TI

In [ ]:
def skenario_ketiga_tversky_index(skenario : object,*, alpha_1 = 0.1, alpha_2 = 0, k_user = 5) :

    k_item = [5,10,15,18,20,25,30,40,50,100,200]

    ti_k_item_fold = []
    ti_mean_k_item_fold = []

    # Mengakses parameter k_item
    for k_item_index in k_item :

        skenario_kedua_k_item = skenario[0.7][k_user][k_item_index][alpha_1][alpha_2]
        ti_mean_k_item_per_fold = []

        for fold in skenario_kedua_k_item :
            # Tahap 1 : menjumlahkan evaluasi setiap fold
            ti_sum_k_item_per_fold = sum(j[99] for j in fold)
            ti_mean_k_item_per_fold += [ti_sum_k_item_per_fold/len(skenario_kedua_k_item)]
        
        # Tahap 3 : menghitung rata-rata setiap fold dengan menjumlahkan nilai evaluasi setiap pengguna dibagi dengan banyaknya pengguna
        mean_k_item_per_fold = sum(ti_mean_k_item_per_fold)/len(skenario_kedua_k_item)
        
        ti_mean_k_item_per_fold += [mean_k_item_per_fold]
        ti_k_item_fold += [ti_mean_k_item_per_fold]
        ti_mean_k_item_fold += [[5,k_item_index,mean_k_item_per_fold]]

    return {
        "mean" : ti_k_item_fold,
        "result" : ti_mean_k_item_fold,
    }

def df_skenario_ketiga_tversky_index(dataProcess, type = "result"):
    import pandas as pd

    k_item = [5,10,15,18,20,25,30,40,50,100,200]

    return pd.DataFrame(dataProcess[type],index=[ f"k_item_{i}" for i in k_item],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])

def visualisation_skenario_ketiga_tversky_index_per_fold(dataProcess, type, title) :
    import matplotlib.pyplot as plt
    k_item = [5,10,15,18,20,25,30,40,50,100,200]

    if type == "mean" :
        # Definisikan markers dan colors
        # markers = ['o', 's', '^', 'D', 'o']
        colors = ['b', 'g', 'r', 'c', 'm']

        plt.figure(figsize=(10, 6))

        for fold_idx in range(5):
            per_fold = [dataProcess[type][k][fold_idx] for k in range(len(k_item))]
            plt.plot(
                k_item,
                per_fold,
                marker="o",
                color=colors[fold_idx],
                label=f'Fold-{fold_idx+1}'
            )
        plt.legend()

    elif type == "result" :
        import pandas as pd
        df = pd.DataFrame(dataProcess[type], index=[i for i in range(1, len(dataProcess[type]) + 1)], columns=["k_user", "k_item", title])

        import matplotlib.pyplot as plt

        plt.figure(figsize=(10, 6))
        plt.plot(k_item, df[title], marker='o')
    
    plt.xlabel(r'$k^{item}$')
    plt.ylabel(title)
    plt.title(title+r' per Fold untuk Setiap $k^{item}$')
    plt.grid(True)
    plt.show()



## Keempat TI

In [ ]:
def skenario_keempat_tversky_index(skenario : object,*, alpha_1 = 0.1, alpha_2 = 0, k_user = 5,k_item) :

    gamma = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]

    ti_gamma_fold = []
    ti_mean_gamma_fold = []

    # Mengakses parameter gamma
    for gamma_index in gamma :

        skenario_kedua_gamma = skenario[gamma_index][k_user][k_item][alpha_1][alpha_2]
        ti_mean_gamma_per_fold = []

        for fold in skenario_kedua_gamma :
            # Tahap 1 : menjumlahkan evaluasi setiap fold
            ti_sum_gamma_per_fold = sum(j[99] for j in fold)
            ti_mean_gamma_per_fold += [ti_sum_gamma_per_fold/len(skenario_kedua_gamma)]
        
        # Tahap 3 : menghitung rata-rata setiap fold dengan menjumlahkan nilai evaluasi setiap pengguna dibagi dengan banyaknya pengguna
        mean_gamma_per_fold = sum(ti_mean_gamma_per_fold)/len(skenario_kedua_gamma)
        
        ti_mean_gamma_per_fold += [mean_gamma_per_fold]
        ti_gamma_fold += [ti_mean_gamma_per_fold]
        ti_mean_gamma_fold += [[5,gamma_index,mean_gamma_per_fold]]

    return {
        "mean" : ti_gamma_fold,
        "result" : ti_mean_gamma_fold,
    }

def df_skenario_keempat_tversky_index(dataProcess, type = "result"):
    import pandas as pd

    gamma = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]

    return pd.DataFrame(dataProcess[type],index=[ f"gamma_{i}" for i in gamma],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])

def visualisation_skenario_keempat_tversky_index_per_fold(dataProcess, type, title) :
    import matplotlib.pyplot as plt
    gamma = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]

    if type == "mean" :
        # Definisikan markers dan colors
        # markers = ['o', 's', '^', 'D', 'o']
        colors = ['b', 'g', 'r', 'c', 'm']

        plt.figure(figsize=(10, 6))

        for fold_idx in range(5):
            per_fold = [dataProcess[type][k][fold_idx] for k in range(len(gamma))]
            plt.plot(
                gamma,
                per_fold,
                marker="o",
                color=colors[fold_idx],
                label=f'Fold-{fold_idx+1}'
            )
        plt.legend()

    elif type == "result" :
        import pandas as pd
        df = pd.DataFrame(dataProcess[type], index=[i for i in range(1, len(dataProcess[type]) + 1)], columns=["k_user", "k_item", title])

        import matplotlib.pyplot as plt

        plt.figure(figsize=(10, 6))
        plt.plot(gamma, df[title], marker='o')
    
    plt.xlabel(r'$k^{item}$')
    plt.ylabel(title)
    plt.title(title+r' per Fold untuk Setiap $k^{item}$')
    plt.grid(True)
    plt.show()

def get_the_highest_value_from_skenario_keempat(skenario,*,alpha_1 = 0.1, alpha_2 = 0, k_user = 5,k_item) :
    
    data = skenario.get_all_skenario()
    data_process = skenario_keempat_tversky_index(data,alpha_1 = alpha_1, alpha_2 = alpha_2, k_user = k_user,k_item=k_item)

    df = df_skenario_keempat_tversky_index(data_process,"mean")
    get_highest_value(df)

    return df

## Pertama DC

In [11]:
def skenario_pertama_dice_coefficient(skenario : object,*,gamma,k_item = 20) :

    k_user = [5,10,15,18,20,25,30,40,50,100,200]

    ti_k_user_fold = []
    ti_mean_k_user_fold = []

    # Mengakses parameter k_user
    for k_user_index in k_user :

        skenario_kedua_k_user = skenario[gamma][k_user_index][k_item]
        ti_mean_k_user_per_fold = []

        for fold in skenario_kedua_k_user :
            # Tahap 1 : menjumlahkan evaluasi setiap fold
            ti_sum_k_user_per_fold = sum(j[99] for j in fold)
            ti_mean_k_user_per_fold += [ti_sum_k_user_per_fold/len(skenario_kedua_k_user)]
        
        # Tahap 3 : menghitung rata-rata setiap fold dengan menjumlahkan nilai evaluasi setiap pengguna dibagi dengan banyaknya pengguna
        mean_k_user_per_fold = sum(ti_mean_k_user_per_fold)/len(skenario_kedua_k_user)
        
        ti_mean_k_user_per_fold += [mean_k_user_per_fold]
        ti_k_user_fold += [ti_mean_k_user_per_fold]
        ti_mean_k_user_fold += [[5,k_user_index,mean_k_user_per_fold]]

    return {
        "mean" : ti_k_user_fold,
        "result" : ti_mean_k_user_fold,
    }

def df_skenario_pertama_dice_coefficient(dataProcess, type = "result"):
    import pandas as pd

    k_user = [5,10,15,18,20,25,30,40,50,100,200]

    return pd.DataFrame(dataProcess[type],index=[ f"k_user_{i}" for i in k_user],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])

def visualisation_skenario_pertama_dice_coefficient_per_fold(dataProcess, type, title) :
    import matplotlib.pyplot as plt
    k_user = [5,10,15,18,20,25,30,40,50,100,200]

    if type == "mean" :
        # Definisikan markers dan colors
        # markers = ['o', 's', '^', 'D', 'o']
        colors = ['b', 'g', 'r', 'c', 'm']

        plt.figure(figsize=(10, 6))

        for fold_idx in range(5):
            per_fold = [dataProcess[type][k][fold_idx] for k in range(len(k_user))]
            plt.plot(
                k_user,
                per_fold,
                marker="o",
                color=colors[fold_idx],
                label=f'Fold-{fold_idx+1}'
            )
        plt.legend()

    elif type == "result" :
        import pandas as pd
        df = pd.DataFrame(dataProcess[type], index=[i for i in range(1, len(dataProcess[type]) + 1)], columns=["k_user", "k_item", title])

        import matplotlib.pyplot as plt

        plt.figure(figsize=(10, 6))
        plt.plot(k_user, df[title], marker='o')
    
    plt.xlabel(r'$k^{user}$')
    plt.ylabel(title)
    plt.title(title+r' per Fold untuk Setiap $k^{user}$')
    plt.grid(True)
    plt.show()

def get_the_highest_value_from_skenario_pertama_dice_coefficient(skenario,*,gamma=0.7,k_item=20) :
    
    data = skenario.get_all_skenario()
    data_process = skenario_pertama_dice_coefficient(data,gamma=gamma,k_item=k_item)

    df = df_skenario_pertama_dice_coefficient(data_process,"mean")
    get_highest_value(df)

    return df

# Recall

### Skenario mencari $\alpha_1$ dan $\alpha_2$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [30],
            "k_item" : [20],
            "alpha_1" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "alpha_2" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        }  

ti_skenario_pertama_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/recall/skenario_pertama",N=100,n=100)

In [ ]:
ti_skenario_pertama_recall.skenario_executed()

In [ ]:
skenario = ti_skenario_pertama_recall.get_all_skenario()

In [ ]:
data_process_recall_skenario_1 = skenario_pertama_tversky_index(skenario)

In [ ]:
import pandas as pd
import itertools
alpha = [round(i*0.1,2) for i in range(11)]

df_recall_skenario_pertama = pd.DataFrame(data_process_recall_skenario_1["mean"],index=[ f"A1_{str(i)}_A2_{str(j)}" for i,j in itertools.product(alpha,alpha)],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])
df_recall_skenario_pertama

In [ ]:
get_highest_value(df_recall_skenario_pertama)

#### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E


param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

dc_skenario_pertama_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_folder="Eksperimen/dice_coefficient/recall/skenario_pertama",N=100,n=100)

Start
Check skenario
Jumlah skenario : 11, Jumlah skenario telah selesai : 11
Semua skenario telah selesai
End


## Skenario mencari $k^{user}$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
            "alpha_1" : [0.1],
            "alpha_2" : [0],
        }

ti_skenario_kedua_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/recall/skenario_kedua",N=100,n=None)

In [ ]:
ti_skenario_kedua_recall.skenario_executed()

In [ ]:
ti_skenario_kedua_recall_data = ti_skenario_kedua_recall.get_all_skenario()

In [ ]:
data_process_recall_ti_kedua = skenario_kedua_tversky_index(ti_skenario_kedua_recall_data)

In [ ]:
import pandas as pd

k_user = [5,10,15,18,20,25,30,40,50,100,200]

df_skenario_kedua_recall = pd.DataFrame(data_process_recall_ti_kedua["mean"],index=[ f"k_user_{i}" for i in k_user],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])
df_skenario_kedua_recall

In [ ]:
get_highest_value(df_skenario_kedua_recall)

In [ ]:
import matplotlib.pyplot as plt

# Definisikan markers dan colors
# markers = ['o', 's', '^', 'D', 'o']
colors = ['b', 'g', 'r', 'c', 'm']

k_user = [5,10,15,18,20,25,30,40,50,100,200]

plt.figure(figsize=(10, 6))

for fold_idx in range(5):
    dcg_per_fold = [data_process_recall_ti_kedua["mean"][k][fold_idx] for k in range(len(k_user))]
    plt.plot(
        k_user,
        dcg_per_fold,
        marker="o",
        color=colors[fold_idx],
        label=f'Fold-{fold_idx+1}'
    )

plt.xlabel(r'$k^{user}$')
plt.ylabel('Recall@20')
plt.title(r'Recall@20 per Fold untuk Setiap $k^{user}$')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd

df = pd.DataFrame(data_process_recall_ti_kedua["result"], index=[i for i in range(1, len(data_process_recall_ti_kedua["result"]) + 1)], columns=["k_user", "k_item", "recall"])

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot([5,10,15,18,20,25,30,40,50,100,200], df["recall"], marker='o')
plt.xlabel(r"$k^{user}$")
plt.ylabel("Recall")
plt.title(r"Recall : $k^{user}$")
plt.grid(True)
plt.show()

## Skenario mencari $k^{item}$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5],
            "k_item" : [5,10,15,18,20,25,30,40,50,100,200],
            "alpha_1" : [0.1],
            "alpha_2" : [0],
        }

ti_skenario_ketiga_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/recall/skenario_ketiga",N=100,n=100)

In [ ]:
ti_skenario_ketiga_recall.skenario_executed()

In [ ]:
skenario_recall_ti_ketiga = ti_skenario_ketiga_recall.get_all_skenario()

In [ ]:
data_process_recall_ti_ketiga = skenario_ketiga_tversky_index(skenario_recall_ti_ketiga,alpha_1=0.1,alpha_2=0,k_user=5)

In [ ]:
df_recall_ti_ketiga = df_skenario_ketiga_tversky_index(data_process_recall_ti_ketiga,type="mean")
df_recall_ti_ketiga

In [ ]:
get_highest_value(df_recall_ti_ketiga)

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_recall_ti_ketiga,"mean","Recall")

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_recall_ti_ketiga,"result","Recall")

### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

dc_skenario_pertama_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_file="Eksperimen/dice_coefficient/recall/FBO_A1_and_A2.joblib",N=100,n=None)

## Skenario mencari $\Gamma$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "k_user" : [5],
            "k_item" : [50],
            "alpha_1" : [0.1],
            "alpha_2" : [0],
        }

ti_skenario_keempat_recall = EksperimenHCF("data/ml-100k",Evaluation=E.Recall,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/recall/skenario_keempat",N=100,n=100)

In [ ]:
ti_skenario_keempat_recall.skenario_executed()

In [ ]:
ti_skenario_keempat_recall_data = ti_skenario_keempat_recall.get_all_skenario()

In [ ]:
data_process_ti_skenario_keempat_recall = skenario_keempat_tversky_index(ti_skenario_keempat_recall_data,alpha_1=0.1,alpha_2=0,k_user=5,k_item=50)

In [ ]:
df_ti_skenario_keempat_recall = df_skenario_keempat_tversky_index(data_process_ti_skenario_keempat_recall,"mean")
df_ti_skenario_keempat_recall 

In [ ]:
get_highest_value(df_ti_skenario_keempat_recall)

In [ ]:
visualisation_skenario_keempat_tversky_index_per_fold(data_process_ti_skenario_keempat_recall,"mean","Recall")

In [ ]:
visualisation_skenario_keempat_tversky_index_per_fold(data_process_ti_skenario_keempat_recall,"result","Recall")

# F1 Score

### Skenario Mencari $\alpha_1$ dan $\alpha_2$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [30],
            "k_item" : [20],
            "alpha_1" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "alpha_2" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        }

ti_skenario_pertama_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/f1_score/skenario_pertama",N=100,n=100)

In [ ]:
ti_skenario_pertama_f1_score.skenario_executed()

In [ ]:
skenario_f1_score_pertama = ti_skenario_pertama_f1_score.get_all_skenario()

In [ ]:
data_process_f1_score_skenario_1 = skenario_pertama_tversky_index(skenario_f1_score_pertama)

In [ ]:
import pandas as pd
import itertools
alpha = [round(i*0.1,2) for i in range(11)]

df_f1_score_skenario_pertama = pd.DataFrame(data_process_f1_score_skenario_1["mean"],index=[ f"A1_{str(i)}_A2_{str(j)}" for i,j in itertools.product(alpha,alpha)],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])
df_f1_score_skenario_pertama

In [ ]:
get_highest_value(df_f1_score_skenario_pertama)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Ambil kolom rata-rata dari ti_mean_gamma_overall_fold_ap
rata_rata = np.array(data_process_f1_score_skenario_1["mean"])[:, -1]

# Ubah menjadi matriks 11x11 sesuai alpha_1 dan alpha_2
heatmap_matrix = rata_rata.reshape(len(alpha), len(alpha))

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_matrix, annot=True, fmt=".2f", cmap="viridis", 
            xticklabels=alpha, yticklabels=alpha, cbar_kws={'label': 'Recall rata-rata'})
plt.xlabel(r'$\alpha_2$')
plt.ylabel(r'$\alpha_1$')
plt.title(r'Heatmap Recall rata-rata: $\alpha_1$ vs $\alpha_2$')
plt.show()


#### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E


param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

dc_skenario_pertama_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_folder="Eksperimen/dice_coefficient/f1_score/skenario_pertama",N=20,n=20)

In [ ]:
dc_skenario_pertama_f1_score.skenario_executed()

### Skenario mencari $k^{user}$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
            "alpha_1" : [0.4],
            "alpha_2" : [0.1],
        }

ti_skenario_kedua_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/f1_score/skenario_kedua",N=100,n=100)

In [ ]:
ti_skenario_kedua_f1_score.skenario_executed()

In [ ]:
ti_f1_score_skenario_kedua_data = ti_skenario_kedua_f1_score.get_all_skenario()

In [ ]:
data_process_f1_score_ti_kedua = skenario_kedua_tversky_index(ti_f1_score_skenario_kedua_data,alpha_1=0.4,alpha_2=0.1)

In [ ]:
df_skenario_kedua_f1_score = df_skenario_kedua_tversky_index(data_process_f1_score_ti_kedua,type="mean")
df_skenario_kedua_f1_score

In [ ]:
get_highest_value(df_skenario_kedua_f1_score)

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_f1_score_ti_kedua, "mean", "AP")

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_f1_score_ti_kedua, "result", "AP")

#### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

skenario_pertama_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_file="Eksperimen/dice_coefficient/f1_score/FBO_A1_and_A2.joblib",N=100,n=None)

## Skenario mencari $k^{item}$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_item" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_user" : [5],
            "alpha_1" : [0.4],
            "alpha_2" : [0.1],
        }

ti_skenario_ketiga_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/f1_score/skenario_ketiga",N=100,n=100)

In [ ]:
ti_skenario_ketiga_f1_score.skenario_executed()

In [ ]:
ti_skenario_ketiga_f1_score_data = ti_skenario_ketiga_f1_score.get_all_skenario()

In [ ]:
data_process_f1_score_ti_ketiga = skenario_ketiga_tversky_index(ti_skenario_ketiga_f1_score_data,alpha_1=0.4,alpha_2=0.1,k_user=5)

In [ ]:
df_f1_score_ti_ketiga = df_skenario_ketiga_tversky_index(data_process_f1_score_ti_ketiga,type="mean")
df_f1_score_ti_ketiga

In [ ]:
get_highest_value(df_f1_score_ti_ketiga)

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_f1_score_ti_ketiga,"mean","F1 Score")

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_f1_score_ti_ketiga,"result","F1 Score")

## Skenario mencari $\Gamma$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "k_user" : [5],
            "k_item" : [18],
            "alpha_1" : [0.4],
            "alpha_2" : [0.1],
        }

ti_skenario_keempat_f1_score = EksperimenHCF("data/ml-100k",Evaluation=E.F1_score,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/f1_score/skenario_keempat",N=100,n=100)

In [ ]:
ti_skenario_keempat_f1_score.skenario_executed()

In [ ]:
data_ti_keempat_f1_score = get_the_highest_value_from_skenario_keempat(ti_skenario_keempat_f1_score,alpha_1=0.4,alpha_2=0.1,k_user=5,k_item=18)
data_ti_keempat_f1_score

# Precision

### Skenario mencari $\alpha_1$ dan $\alpha_2$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [30],
            "k_item" : [20],
            "alpha_1" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "alpha_2" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        }

ti_skenario_pertama_precision = EksperimenHCF("data/ml-100k",Evaluation=E.Precision,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/precision/skenario_pertama",N=100,n=None)

In [ ]:
ti_skenario_pertama_precision.skenario_executed()

In [ ]:
skenario = ti_skenario_pertama_precision.get_all_skenario()

In [ ]:
data_process_skenario_pertama_precision= skenario_pertama_tversky_index(skenario)

In [ ]:
import pandas as pd
import itertools
alpha = [round(i*0.1,2) for i in range(11)]

df = pd.DataFrame(data_process_skenario_pertama_precision["mean"],index=[ f"A1_{str(i)}_A2_{str(j)}" for i,j in itertools.product(alpha,alpha)],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])

In [ ]:
df

In [ ]:
get_highest_value(df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Ambil kolom rata-rata dari ti_mean_gamma_overall_fold_ap
rata_rata = np.array(data_process_skenario_pertama_precision["mean"])[:, -1]

# Ubah menjadi matriks 11x11 sesuai alpha_1 dan alpha_2
heatmap_matrix = rata_rata.reshape(len(alpha), len(alpha))

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_matrix, annot=True, fmt=".2f", cmap="viridis", 
            xticklabels=alpha, yticklabels=alpha, cbar_kws={'label': 'Precision rata-rata'})
plt.xlabel(r'$\alpha_2$')
plt.ylabel(r'$\alpha_1$')
plt.title(r'Heatmap Precision rata-rata: $\alpha_1$ vs $\alpha_2$')
plt.show()


### Skenario mencari $k^{user}$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
            "alpha_1" : [0.1],
            "alpha_2" : [0],
        }

ti_skenario_kedua_precision = EksperimenHCF("data/ml-100k",Evaluation=E.Precision,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/precision/skenario_kedua",N=100,n=100)

In [ ]:
ti_skenario_kedua_precision.skenario_executed()

In [ ]:
ti_skenario_kedua_precision_data = ti_skenario_kedua_precision.get_all_skenario()

In [ ]:
data_process_precision_ti_kedua = skenario_kedua_tversky_index(ti_skenario_kedua_precision_data)

In [ ]:
df_ti_skenario_kedua_precision = df_skenario_kedua_tversky_index(data_process_precision_ti_kedua,type="mean")
df_ti_skenario_kedua_precision

In [ ]:
get_highest_value(df_ti_skenario_kedua_precision)

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_precision_ti_kedua, "mean", "Precision")

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_precision_ti_kedua, "result", "Precision")

#### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

skenario_pertama_precision = EksperimenHCF("data/ml-100k",Evaluation=E.Precision,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_file="Eksperimen/dice_coefficient/precision/FBO_A1_and_A2.joblib",N=100,n=None)

## Skenario mencari $k^{item}$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5],
            "k_item" : [5,10,15,18,20,25,30,40,50,100,200],
            "alpha_1" : [0.4],
            "alpha_2" : [0.1],
        }

ti_skenario_ketiga_precision = EksperimenHCF("data/ml-100k",Evaluation=E.Precision,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/precision/skenario_ketiga",N=100,n=100)

In [ ]:
ti_skenario_ketiga_precision.skenario_executed()

In [ ]:
ti_skenario_ketiga_precision_data = ti_skenario_ketiga_precision.get_all_skenario()

In [ ]:
data_process_ti_skenario_ketiga_precision = skenario_ketiga_tversky_index(ti_skenario_ketiga_precision_data,alpha_1=0.4,alpha_2=0.1,k_user=5)

In [ ]:
df_ti_skenario_ketiga_precision = df_skenario_ketiga_tversky_index(data_process_ti_skenario_ketiga_precision,"mean")
df_ti_skenario_ketiga_precision 

In [ ]:
get_highest_value(df_ti_skenario_ketiga_precision)

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_ti_skenario_ketiga_precision,"mean","Precision")

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_ti_skenario_ketiga_precision,"result","Precision")

## Skenario mencari $\Gamma$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "k_user" : [5],
            "k_item" : [18],
            "alpha_1" : [0.4],
            "alpha_2" : [0.1],
        }

ti_skenario_keempat_precision = EksperimenHCF("data/ml-100k",Evaluation=E.Precision,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/precision/skenario_keempat",N=100,n=100)

In [ ]:
data_ti_keempat_precision = get_the_highest_value_from_skenario_keempat(ti_skenario_keempat_precision,alpha_1=0.4,alpha_2=0.1,k_user=5,k_item=18)
data_ti_keempat_precision

# AP

### Skenario Mencari $\alpha_1$ dan $\alpha_2$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [30],
            "k_item" : [20],
            "alpha_1" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "alpha_2" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
        }

ti_skenario_pertama_ap = EksperimenHCF("data/ml-100k",Evaluation=E.AP,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/ap/skenario_pertama",N=100,n=100)

In [ ]:
ti_skenario_pertama_ap.skenario_executed()

In [ ]:
ap_skenario_1 = ti_skenario_pertama_ap.get_all_skenario()

In [ ]:
data_process_f1_score_skenario_1 = skenario_pertama_tversky_index(ap_skenario_1)

In [ ]:
import pandas as pd
import itertools
alpha = [round(i*0.1,2) for i in range(11)]

df_ap_skenario_pertama = pd.DataFrame(data_process_f1_score_skenario_1["mean"],index=[ f"A1_{str(i)}_A2_{str(j)}" for i,j in itertools.product(alpha,alpha)],columns=[f"Fold-{i}" if i != 6 else "rata-rata" for i in range(1,7)])
df_ap_skenario_pertama

In [ ]:
get_highest_value(df_ap_skenario_pertama)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Ambil kolom rata-rata dari ti_mean_gamma_overall_fold_ap
rata_rata = np.array(data_process_f1_score_skenario_1["mean"])[:, -1]

# Ubah menjadi matriks 11x11 sesuai alpha_1 dan alpha_2
heatmap_matrix = rata_rata.reshape(len(alpha), len(alpha))

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_matrix, annot=True, fmt=".2f", cmap="viridis", 
            xticklabels=alpha, yticklabels=alpha, cbar_kws={'label': 'AP rata-rata'})
plt.xlabel(r'$\alpha_2$')
plt.ylabel(r'$\alpha_1$')
plt.title(r'Heatmap AP rata-rata: $\alpha_1$ vs $\alpha_2$')
plt.show()


### Skenario mencari $k^{user}$

#### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
            "alpha_1" : [0.8],
            "alpha_2" : [0.2],
        }

ti_skenario_kedua_AP = EksperimenHCF("data/ml-100k",Evaluation=E.AP,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/ap/skenario_kedua",N=100,n=100)

In [ ]:
ti_skenario_kedua_AP.skenario_executed()

In [ ]:
data_process_ap_skenario_kedua = ti_skenario_kedua_AP.get_all_skenario()

In [ ]:
data_process_AP_ti_kedua = skenario_kedua_tversky_index(data_process_ap_skenario_kedua,alpha_1=0.8,alpha_2=0.2)

In [ ]:
df_ti_skenario_kedua_ap = df_skenario_kedua_tversky_index(data_process_AP_ti_kedua,type="mean")
df_ti_skenario_kedua_ap


In [ ]:
get_highest_value(df_ti_skenario_kedua_ap)

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_AP_ti_kedua, "mean", "AP")

In [ ]:
visualisation_skenario_kedua_tversky_index_per_fold(data_process_AP_ti_kedua, "result", "AP")

#### Dice Coefficient

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [5,10,15,18,20,25,30,40,50,100,200],
            "k_item" : [20],
        }

skenario_pertama_ap = EksperimenHCF("data/ml-100k",Evaluation=E.AP,object=S.DC,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],path_file="Eksperimen/dice_coefficient/ap/FBO_A1_and_A2.joblib",N=100,n=None)

## Skenario Mencari $k^{item}$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0.7],
            "k_user" : [30],
            "k_item" : [5,10,15,18,20,25,30,40,50,100,200],
            "alpha_1" : [0.8],
            "alpha_2" : [0.2],
        }

ti_skenario_ketiga_AP = EksperimenHCF("data/ml-100k",Evaluation=E.AP,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/ap/skenario_ketiga",N=100,n=100)

In [ ]:
ti_skenario_ketiga_AP.skenario_executed()

In [ ]:
ti_skenario_ketiga_AP_data = ti_skenario_ketiga_AP.get_all_skenario()

In [ ]:
data_process_ti_skenario_ketiga_AP = skenario_ketiga_tversky_index(ti_skenario_ketiga_AP_data,alpha_1=0.8,alpha_2=0.2,k_user=30)

In [ ]:
df_ti_skenario_ketiga_AP = df_skenario_ketiga_tversky_index(data_process_ti_skenario_ketiga_AP,"mean")
df_ti_skenario_ketiga_AP 

In [ ]:
get_highest_value(df_ti_skenario_ketiga_AP)

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_ti_skenario_ketiga_AP,"mean","AP")

In [ ]:
visualisation_skenario_ketiga_tversky_index_per_fold(data_process_ti_skenario_ketiga_AP,"result","AP")

## Skenario mencari $\Gamma$

### Tversky Index

In [ ]:
from Skenario import EksperimenHCF
import DistanceBased.similarities as S
import Evaluation as E

param = {
            "gamma" : [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
            "k_user" : [30],
            "k_item" : [20],
            "alpha_1" : [0.8],
            "alpha_2" : [0.2],
        }

ti_skenario_keempat_ap = EksperimenHCF("data/ml-100k",Evaluation=E.AP,object=S.TI,k_user=param["k_user"],k_item=param["k_item"],gamma=param["gamma"],alpha_1=param["alpha_1"],alpha_2=param["alpha_2"],path_folder="Eksperimen/tversky_index/ap/skenario_keempat",N=100,n=100)

In [ ]:
data_ti_keempat_ap = get_the_highest_value_from_skenario_keempat(ti_skenario_keempat_ap,alpha_1=0.8,alpha_2=0.2,k_user=30,k_item=20)
data_ti_keempat_ap